**Cell B1 — Setup**

In [ ]:
# ============================================================
# CELL B1: SETUP (Model B — CNN U-Net for ensemble)
# ============================================================

import os

# Drive mount
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

# MONAI + dependencies install
print("\n📦 Packages install করছি...")
os.system("pip install monai -q")
os.system("pip install einops -q")
print("✅ MONAI installed")

# Sanity check
DATA_ROOT = "/content/drive/MyDrive/GAVE2_preliminary"
print("\n--- Sanity Check ---")
print(f"Dataset: {'✅' if os.path.exists(DATA_ROOT) else '❌'}")

# RETFound predictions আছে কিনা (ensemble-এর জন্য পরে লাগবে)
#pred_dir = f"{DATA_ROOT}/retfound_val_preds"
#if os.path.exists(pred_dir):
 #   n_preds = len([f for f in os.listdir(pred_dir) if f.endswith('.npy')])
 #   print(f"RETFound predictions: ✅ {n_preds}টা saved")
#else:
   # print("RETFound predictions: ⚠️ পাওয়া যায়নি")

# training images
train_img = f"{DATA_ROOT}/training/images"
if os.path.exists(train_img):
    print(f"Training images: ✅ {len(os.listdir(train_img))}টা case")

print("\n🎯 Cell B1 সম্পন্ন। Cell B2 run করো।")

**Cell B2 — Imports + Paths**

In [ ]:
# ============================================================
# CELL B2: IMPORTS + PATHS (Model B)
# ============================================================

import os, sys, random, math
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from monai.networks.nets import FlexibleUNet

# Reproducibility
SEED = 123   # RETFound-এ 42 ছিল — আলাদা seed দিলে ensemble diversity বাড়ে
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
print("✅ Imports সম্পন্ন")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Paths
DATA_ROOT  = "/content/drive/MyDrive/GAVE2_preliminary"
TRAIN_DIR  = f"{DATA_ROOT}/training"
VAL_DIR    = f"{DATA_ROOT}/validation"
SAVE_PATH  = f"{DATA_ROOT}/best_model_cnn.pth"   

# Hyperparameters
IMG_SIZE     = 1024      
BATCH_SIZE   = 2         # CNN হালকা, batch 2 দেওয়া যায়
NUM_WORKERS  = 2
NUM_EPOCHS   = 70
LR           = 3e-4      # CNN scratch+pretrained encoder
WEIGHT_DECAY = 1e-3
NUM_CLASSES  = 4
CLASS_WEIGHTS = torch.tensor([0.3, 4.0, 3.0, 6.0]).to(DEVICE)  

print(f"\n--- Config ---")
print(f"  IMG_SIZE={IMG_SIZE}, BATCH={BATCH_SIZE}, EPOCHS={NUM_EPOCHS}")
print(f"  Save: {SAVE_PATH}")
print(f"  Seed: {SEED} ")
print("\n🎯 Cell B2 সম্পন্ন। Cell B3 run করো।")

**Cell B3 — Dataset**

In [ ]:
# ============================================================
# CELL B3: DATASET (Model B — 5-channel concat input)
# ============================================================

def parse_av_label(label_path):
    """Color-coded PNG → class index mask (H,W)"""
    img = np.array(Image.open(label_path).convert("RGB"))
    R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]
    mask = np.zeros(img.shape[:2], dtype=np.uint8)
    mask[(R>150) & (G<50)  & (B<50)]  = 1  # artery
    mask[(R<50)  & (G<50)  & (B>150)] = 2  # vein
    mask[(R<50)  & (G>150) & (B<50)]  = 3  # overlap
    return mask


class GAVE2DatasetCNN(Dataset):
    """
    Model B dataset — CFP + FFA_diff + FFA_A → 5-channel input।
    RETFound-এর সাথে পার্থক্য: সব একসাথে concat (আলাদা encoder নেই)।
    """
    def __init__(self, case_ids, augment=False):
        self.case_ids = case_ids
        self.augment  = augment

    def __len__(self):
        return len(self.case_ids)

    def _load_rgb(self, path):
        img = Image.open(path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2,0,1)  # (3,H,W)

    def _load_gray(self, path):
        img = Image.open(path).convert("L")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).unsqueeze(0)  # (1,H,W)

    def __getitem__(self, idx):
        case = f"g_{self.case_ids[idx]+1:03d}"
        base = TRAIN_DIR

        cfp    = self._load_rgb(f"{base}/images/{case}.png")    # (3,H,W)
        ffa_a  = self._load_gray(f"{base}/FFA_A/{case}.png")    # (1,H,W)
        ffa_av = self._load_gray(f"{base}/FFA_AV/{case}.png")   # (1,H,W)

        # label
        label = parse_av_label(f"{base}/av/{case}.png")
        label_pil = Image.fromarray(label).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
        label = torch.from_numpy(np.array(label_pil)).long()

        # FFA diff (novelty signal — Model B-তেও রাখছি)
        ffa_diff = torch.abs(ffa_av - ffa_a)  # (1,H,W)

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                cfp=TF.hflip(cfp); ffa_a=TF.hflip(ffa_a); ffa_diff=TF.hflip(ffa_diff); label=TF.hflip(label)
            if random.random() > 0.5:
                cfp=TF.vflip(cfp); ffa_a=TF.vflip(ffa_a); ffa_diff=TF.vflip(ffa_diff); label=TF.vflip(label)
            if random.random() > 0.5:
                angle = random.uniform(-15, 15)
                cfp=TF.rotate(cfp,angle); ffa_a=TF.rotate(ffa_a,angle); ffa_diff=TF.rotate(ffa_diff,angle)
                label=TF.rotate(label.unsqueeze(0),angle,
                       interpolation=TF.InterpolationMode.NEAREST).squeeze(0)
            if random.random() > 0.5:
                cfp = T.ColorJitter(brightness=0.2, contrast=0.2)(cfp)

        # CFP normalize (ImageNet — EfficientNet pretrained এভাবে)
        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        cfp = (cfp - mean) / std

        # --- 5-channel concat: CFP(3) + FFA_diff(1) + FFA_A(1) ---
        x = torch.cat([cfp, ffa_diff, ffa_a], dim=0)  # (5,H,W)

        return {"input": x, "label": label, "case": case}


# --- Test ---
print("Dataset test করছি...")
train_idx = list(range(40))
val_idx   = list(range(40, 50))

train_ds = GAVE2DatasetCNN(train_idx, augment=True)
val_ds   = GAVE2DatasetCNN(val_idx,   augment=False)

sample = train_ds[0]
print(f"✅ input shape: {sample['input'].shape}  (5 channel: CFP3+diff1+ffaA1)")
print(f"✅ label shape: {sample['label'].shape}  unique={sample['label'].unique().tolist()}")
print(f"✅ case: {sample['case']}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
print(f"\n✅ Batch input: {batch['input'].shape}")
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)}")
print("\n🎯 Cell B3 সম্পন্ন। Cell B4 run করো।")

Dataset test করছি...
✅ input shape: torch.Size([5, 1024, 1024])  (5 channel: CFP3+diff1+ffaA1)
✅ label shape: torch.Size([1024, 1024])  unique=[0, 1, 2, 3]
✅ case: g_001

✅ Batch input: torch.Size([2, 5, 1024, 1024])
   Train: 40 | Val: 10

🎯 Cell B3 সম্পন্ন। Cell B4 run করো।


**Cell B4 — Loss Functions**

In [ ]:
# ============================================================
# CELL B4: LOSS FUNCTIONS
# ============================================================

class DiceLoss(nn.Module):
    def __init__(self, num_classes=4, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes; self.smooth = smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        t_oh = F.one_hot(targets, self.num_classes).permute(0,3,1,2).float()
        dims = (0,2,3)
        inter = (probs*t_oh).sum(dims); union = probs.sum(dims)+t_oh.sum(dims)
        return 1.0 - ((2*inter+self.smooth)/(union+self.smooth)).mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()


class TverskyLoss(nn.Module):
    def __init__(self, num_classes=4, alpha=0.3, beta=0.7, smooth=1.0):
        super().__init__()
        self.num_classes=num_classes; self.alpha=alpha; self.beta=beta; self.smooth=smooth
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        t_oh = F.one_hot(targets, self.num_classes).permute(0,3,1,2).float()
        dims = (0,2,3)
        TP = (probs*t_oh).sum(dims)
        FP = (probs*(1-t_oh)).sum(dims)
        FN = ((1-probs)*t_oh).sum(dims)
        tversky = (TP+self.smooth)/(TP+self.alpha*FP+self.beta*FN+self.smooth)
        return 1.0 - tversky[1:].mean()


def soft_skeletonize(mask, iters=5):
    eroded = mask.clone()
    for _ in range(iters):
        eroded = -F.max_pool2d(-eroded, 3, 1, 1)
    return F.relu(mask - eroded)


class clDiceLoss(nn.Module):
    def __init__(self, smooth=1.0, iters=5):
        super().__init__()
        self.smooth=smooth; self.iters=iters
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        total = 0.0
        for cls in [1, 2]:
            p = probs[:, cls:cls+1]
            t = (targets==cls).float().unsqueeze(1)
            sp = soft_skeletonize(p, self.iters)
            st = soft_skeletonize(t, self.iters)
            tprec = ((sp*t).sum()+self.smooth)/(sp.sum()+self.smooth)
            tsens = ((st*p).sum()+self.smooth)/(st.sum()+self.smooth)
            total += 1.0 - 2.0*tprec*tsens/(tprec+tsens+1e-8)
        return total/2.0
class BoundaryLoss(nn.Module):
    """
    রক্তনালীর প্রান্ত (boundary) তীক্ষ্ণ করে।

    ধারণা: prediction আর GT-র boundary (edge) বের করে তুলনা করে।
    Boundary = mask - eroded(mask) → পাতলা প্রান্ত রেখা।
    পাতলা রক্তনালীতে প্রান্ত precision DSC ও topology বাড়ায়।
    """
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def _boundary(self, mask):
        # erosion = -maxpool(-mask), boundary = mask - eroded
        eroded = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
        return F.relu(mask - eroded)

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        total = 0.0
        for cls in [1, 2]:  # artery, vein
            p = probs[:, cls:cls+1]                      # (B,1,H,W)
            t = (targets==cls).float().unsqueeze(1)      # (B,1,H,W)
            pb = self._boundary(p)
            tb = self._boundary(t)
            # boundary Dice
            inter = (pb*tb).sum()
            union = pb.sum() + tb.sum()
            total += 1.0 - (2*inter + self.smooth)/(union + self.smooth)
        return total / 2.0

class CombinedLoss(nn.Module):
    """
    নতুন mix: Focal + Tversky + clDice + Boundary
    Boundary যোগ করায় বাকিদের ওজন সামান্য কমিয়ে rebalance।
    """
    def __init__(self, class_weights):
        super().__init__()
        self.focal    = FocalLoss(gamma=2.0, weight=class_weights)
        self.tversky  = TverskyLoss(num_classes=NUM_CLASSES, alpha=0.3, beta=0.7)
        self.cldice   = clDiceLoss(iters=5)
        self.boundary = BoundaryLoss()
    def forward(self, logits, targets):
        lf = self.focal(logits, targets)
        lt = self.tversky(logits, targets)
        lc = self.cldice(logits, targets)
        lb = self.boundary(logits, targets)
        # Focal 0.2, Tversky 0.2, clDice 0.4, Boundary 0.2
        total = 0.2*lf + 0.2*lt + 0.4*lc + 0.2*lb
        return total, lf, lt, lc   # return signature একই (train loop ভাঙবে না)


criterion = CombinedLoss(CLASS_WEIGHTS).to(DEVICE)
with torch.no_grad():
    dl = torch.randn(2, NUM_CLASSES, 64, 64).to(DEVICE)
    dt = torch.randint(0, NUM_CLASSES, (2,64,64)).to(DEVICE)
    tot, lf, lt, lc = criterion(dl, dt)
print(f"✅ Loss: total={tot.item():.4f} focal={lf.item():.4f} tversky={lt.item():.4f} clDice={lc.item():.4f}")
print("🎯 Cell B4 (boundary সহ) সম্পন্ন।")

✅ Loss: total=1.6899 focal=5.4332 tversky=0.7469 clDice=0.7466
🎯 Cell B4 (boundary সহ) সম্পন্ন।


**Cell B5 — Metric**

In [ ]:
# ============================================================
# CELL B5: DICE METRIC
# ============================================================

def compute_dice_scores(logits, targets, num_classes=NUM_CLASSES, smooth=1e-6):
    preds = logits.argmax(dim=1)
    scores = {}; vessel = []
    for cls, name in [(1,'artery'),(2,'vein'),(3,'overlap')]:
        pm = (preds==cls).float(); tm = (targets==cls).float()
        inter = (pm*tm).sum(); union = pm.sum()+tm.sum()
        dice = (2*inter+smooth)/(union+smooth)
        scores[f'{name}_dice'] = dice.item()
        if cls in [1,2]: vessel.append(dice.item())
    scores['mean_dice'] = sum(vessel)/len(vessel)
    return scores

print("✅ Metric function ready")
print("🎯 Cell B5 সম্পন্ন। Cell B6 run করো।")

✅ Metric function ready
🎯 Cell B5 সম্পন্ন। Cell B6 run করো।


**Cell B6 — Model (FlexibleUNet, EfficientNet-B)**

In [ ]:
# ============================================================
# CELL B6: MODEL — MONAI FlexibleUNet (EfficientNet-B5 encoder)
# 5-channel input → 4-class output
# ============================================================

from monai.networks.nets import FlexibleUNet

def build_model():
    """
    FlexibleUNet:
    - backbone: efficientnet-b4 (ImageNet pretrained encoder)
    - in_channels=5: CFP(3) + FFA_diff(1) + FFA_A(1)
    - out_channels=4: BG/artery/vein/overlap
    - pretrained=True: encoder ImageNet weights দিয়ে শুরু
    """
    model = FlexibleUNet(
        in_channels=5,
        out_channels=NUM_CLASSES,
        backbone="efficientnet-b5",
        pretrained=True,          # encoder pretrained
        decoder_channels=(256, 128, 64, 32, 16),
        spatial_dims=2,
    )
    return model


print("Model তৈরি করছি (efficientnet-b5)...")
model = build_model().to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Total params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")

# --- Forward test (VRAM check) ---
print("\nForward pass test (batch=2, 1024)...")
torch.cuda.reset_peak_memory_stats()
model.train()
with torch.cuda.amp.autocast():
    dummy = torch.randn(2, 5, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out = model(dummy)
print(f"✅ Output shape: {out.shape}  (expected: 2,4,1024,1024)")
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"✅ Peak VRAM (forward): {peak:.1f} GB")

del dummy, out
torch.cuda.empty_cache()
print("\n🎯 Cell B6 সম্পন্ন। Cell B7 run করো।")

**Cell B7 — Train / Validate / TTA**

In [ ]:
# ============================================================
# CELL B7: TRAIN / VALIDATE / TTA
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        x      = batch['input'].to(device)   # (B,5,H,W)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss, _, _, _ = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_art, all_vein = [], []
    for batch in loader:
        x      = batch['input'].to(device)
        labels = batch['label'].to(device)
        logits = model(x)
        loss, _, _, _ = criterion(logits, labels)
        total_loss += loss.item()
        s = compute_dice_scores(logits, labels)
        all_art.append(s['artery_dice']); all_vein.append(s['vein_dice'])
    art = sum(all_art)/len(all_art); vein = sum(all_vein)/len(all_vein)
    return total_loss/len(loader), art, vein, (art+vein)/2


@torch.no_grad()
def tta_validate(model, loader, device):
    """4-flip TTA, softmax CPU-তে জমা"""
    model.eval()
    all_art, all_vein = [], []
    for batch in loader:
        x      = batch['input'].to(device)
        labels = batch['label'].to(device)
        preds_sum = torch.zeros(x.shape[0], NUM_CLASSES, IMG_SIZE, IMG_SIZE)
        for hf, vf in [(False,False),(True,False),(False,True),(True,True)]:
            xi = x.clone()
            if hf: xi = torch.flip(xi, [3])
            if vf: xi = torch.flip(xi, [2])
            lg = model(xi)
            if hf: lg = torch.flip(lg, [3])
            if vf: lg = torch.flip(lg, [2])
            preds_sum += F.softmax(lg, dim=1).cpu()
        s = compute_dice_scores(preds_sum.to(device), labels)
        all_art.append(s['artery_dice']); all_vein.append(s['vein_dice'])
    art = sum(all_art)/len(all_art); vein = sum(all_vein)/len(all_vein)
    return art, vein, (art+vein)/2


# --- Smoke test ---
print("Smoke test করছি...")
dummy_ds = GAVE2DatasetCNN([0,1], augment=False)
dummy_loader = DataLoader(dummy_ds, batch_size=2, shuffle=False, num_workers=0)
opt_test = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler_test = torch.amp.GradScaler('cuda')

tr = train_epoch(model, dummy_loader, criterion, opt_test, DEVICE, scaler_test)
vl, art, vein, mean = validate_epoch(model, dummy_loader, criterion, DEVICE)
print(f"✅ train: {tr:.4f} | val: {vl:.4f} | Mean Dice: {mean:.4f}")
print("\n🎯 Cell B7 সম্পন্ন। Cell B8 run করো।")

Smoke test করছি...
✅ train: 0.8286 | val: 0.8324 | Mean Dice: 0.0000

🎯 Cell B7 সম্পন্ন। Cell B8 run করো।


**Cell B8 — Training Loop**

In [ ]:
# ============================================================
# CELL B8: run_training()
# ============================================================

def run_training(num_epochs=NUM_EPOCHS, resume_path=None):
    train_ds = GAVE2DatasetCNN(list(range(40)),     augment=True)
    val_ds   = GAVE2DatasetCNN(list(range(40, 50)), augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    net = build_model().to(DEVICE)
    if resume_path and os.path.exists(resume_path):
        net.load_state_dict(torch.load(resume_path, map_location=DEVICE, weights_only=False))
        print(f"✅ Resumed: {resume_path}")

    criterion = CombinedLoss(CLASS_WEIGHTS).to(DEVICE)
    optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')

    best_dice, best_epoch = 0.0, 0
    print(f"\nTraining: {num_epochs} epochs, batch={BATCH_SIZE}, res={IMG_SIZE}")
    print(f"Model: EfficientNet-B4 U-Net | Save: {SAVE_PATH}")
    print(f"\n{'Ep':>4} | {'TrLoss':>7} | {'ArtDice':>8} | {'VnDice':>7} | {'MnDice':>7} | {'LR':>9}")
    print("-"*58)

    for epoch in range(1, num_epochs + 1):
        tr_loss = train_epoch(net, train_loader, criterion, optimizer, DEVICE, scaler)

        do_val = (epoch % 3 == 0) or (epoch > 30) or (epoch == 1)
        if do_val:
            _, art, vein, mean = validate_epoch(net, val_loader, criterion, DEVICE)
        scheduler.step()
        torch.cuda.empty_cache()
        lr = scheduler.get_last_lr()[0]

        if do_val:
            is_best = mean > best_dice
            if is_best:
                best_dice, best_epoch = mean, epoch
                torch.save(net.state_dict(), SAVE_PATH)
            if epoch % 5 == 0 or epoch == 1 or is_best:
                marker = " ← best" if is_best else ""
                print(f"{epoch:>4} | {tr_loss:>7.4f} | {art:>8.4f} | "
                      f"{vein:>7.4f} | {mean:>7.4f} | {lr:>9.6f}{marker}")

    print(f"\n{'='*58}")
    print(f"শেষ! Best val Dice: {best_dice:.4f} (epoch {best_epoch})")
    net.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False))
    ta, tv, tm = tta_validate(net, val_loader, DEVICE)
    print(f"TTA → Artery: {ta:.4f} | Vein: {tv:.4f} | Mean: {tm:.4f}")
    print(f"{'='*58}")
    return net, tm

print("🎯 Cell B8 সম্পন্ন। Cell B9 run করলে training শুরু।")

🎯 Cell B8 সম্পন্ন। Cell B9 run করলে training শুরু।


**Cell B9 — Training Start**

In [ ]:
# ============================================================
# CELL B9: TRAINING START
# ============================================================

trained_cnn, final_dice = run_training(num_epochs=NUM_EPOCHS)


Training: 70 epochs, batch=2, res=1024
Model: EfficientNet-B4 U-Net | Save: /content/drive/MyDrive/GAVE2_preliminary/best_model_cnn.pth

  Ep |  TrLoss |  ArtDice |  VnDice |  MnDice |        LR
----------------------------------------------------------
   1 |  0.7698 |   0.0557 |  0.0758 |  0.0658 |  0.000300 ← best
   6 |  0.5140 |   0.0807 |  0.0828 |  0.0817 |  0.000295 ← best
   9 |  0.4458 |   0.3898 |  0.4210 |  0.4054 |  0.000288 ← best
  12 |  0.4149 |   0.5604 |  0.5836 |  0.5720 |  0.000279 ← best
  15 |  0.3937 |   0.6126 |  0.6624 |  0.6375 |  0.000267 ← best
  18 |  0.3804 |   0.6870 |  0.7373 |  0.7121 |  0.000254 ← best
  21 |  0.3717 |   0.6912 |  0.7374 |  0.7143 |  0.000238 ← best
  24 |  0.3689 |   0.7146 |  0.7502 |  0.7324 |  0.000221 ← best
  27 |  0.3578 |   0.7213 |  0.7648 |  0.7430 |  0.000203 ← best
  30 |  0.3543 |   0.7254 |  0.7713 |  0.7483 |  0.000184 ← best
  33 |  0.3490 |   0.7277 |  0.7734 |  0.7505 |  0.000164 ← best
  34 |  0.3480 |   0.7290 |  0